# Lab 1 — Data Ingestion: Connect Your Data Sources

### Databricks Data Camp for Business Users 🛍️

**Story so far:** As the new CEO of Databricks Retail Corp., you've set up your
workspace (Lab 0). Your sales data currently lives in an **operational database**
— a **Lakebase** (Databricks-managed Postgres) instance that powers the online
store. On top of that, your strategy team has handed you a **PDF of external
market research**.

To become data-driven, you first need to **get all of this into Databricks**.

In this lab you will:
1. Understand the **two kinds of sources** you're ingesting (a database + files).
2. Use **Lakeflow Connect** to ingest the Lakebase Postgres tables into Unity Catalog.
3. Bring the **market-research PDF** into Databricks.
4. Confirm everything landed in your **bronze** layer.

⏱️ *Estimated time: 25–30 minutes.*

## 1.0 — Load config

Run this first in every lab. It reconnects you to the shared catalog/schema names
from `setup/config.py`.

In [ ]:
import os, sys
def find_repo_root():
    p = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "setup")) and os.path.isdir(os.path.join(p, "labs")):
            return p
        p = os.path.dirname(p)
    return os.getcwd()
REPO_ROOT = find_repo_root()
sys.path.insert(0, os.path.join(REPO_ROOT, "setup"))
from config import get_config
cfg = get_config()
spark.sql(f"USE CATALOG {cfg['CATALOG']}")
print("Catalog:", cfg["CATALOG"], "| Bronze:", cfg["CATALOG_BRONZE"])
print("Lakebase instance:", cfg["LAKEBASE_INSTANCE"])

## 1.1 — What is Lakeflow Connect? (plain-English version)

Think of your data as living in different "buildings":

- 🏢 **The operational database (Lakebase Postgres)** — this is where your live
  store writes every order and customer. It's great for *running* the store, but
  not for *analyzing* it.
- 📄 **Files** — spreadsheets, and in our case a **PDF** of market research.

**Lakeflow Connect** is Databricks' built-in way to **bring data from those other
buildings into your lakehouse** — reliably and on a schedule — without you writing
plumbing code. You point it at a source, pick the tables, and it lands them in
Unity Catalog (your **bronze** layer = the raw, faithful copy).

> **Medallion architecture preview:** we land raw data in **bronze**, clean/join it
> into **silver** (Lab 3), then build business-ready **gold** tables (Lab 3). Bronze
> = "as received."

In **Lab 0** you provisioned and seeded the **Lakebase** source, and you confirmed
your **bronze schema is empty**. This lab is where you fill it — using the *proper*
ingestion path you'd use in real life, rather than pre-loading.

## 1.2 — Ingest with Lakeflow Connect (guided UI steps)

This is the part you do **in the Databricks UI** — it's the click-through a
business user would actually use.

### 1.2a — First, create the Postgres *connection*
Lakeflow Connect ingests **through a Unity Catalog Connection** — a saved,
governed pointer to your Lakebase instance. The Lakebase database won't appear in
the wizard until this connection exists, so create it once.

**Grab the connection details** by running the helper cell just below this one
(1.2-helper). It prints the **host**, **port**, **database**, **user**, and a
short-lived **token** to use as the password.

Then create the connection — **Catalog → External Data → Connections →
Create connection** (or click **Create connection** inside the ingestion wizard):

| Field | Value |
|-------|-------|
| **Connection name** | `retail_corp_lakebase_conn` |
| **Connection type** | PostgreSQL |
| **Host** | the `read_write_dns` from the helper cell |
| **Port** | `5432` |
| **User** | your Databricks username (from the helper cell) |
| **Password** | the **token** printed by the helper cell |

> ⏱️ **The token is short-lived (~1 hour).** Create the connection and run the
> pipeline promptly. If ingestion later fails to authenticate, re-run the helper
> cell for a fresh token and update the connection's password.

### 1.2b — Then run the ingestion wizard
1. In the left sidebar, click **Data Ingestion** (or **+ New → Add data**).
2. Choose **Lakeflow Connect**, then pick the **PostgreSQL / Lakebase** source.
3. **Connection:** select the **`retail_corp_lakebase_conn`** you just created,
   then choose database **`retail_corp`**.
4. **Select the tables to ingest** — choose all six.
5. **Set the primary key and cursor column for each table.** Lakeflow Connect uses
   these to ingest *incrementally*: the **primary key** de-duplicates / merges rows,
   and the **cursor column** is how it detects new or changed rows since the last
   run. Use these values (they match how the source tables are keyed):

   | Table | Primary key | Cursor column |
   |-------|-------------|---------------|
   | `dim_product` | `product_id` | `product_id` |
   | `dim_customer` | `customer_id` | `customer_id` |
   | `fact_orders` | `order_id` | `order_ts` |
   | `fact_order_items` | `order_item_id` | `order_item_id` |
   | `fact_marketing_campaigns` | `campaign_id` | `campaign_id` |
   | `fact_sales_forecast` | `forecast_id` | `forecast_id` |

   > 💡 **Why these?** `fact_orders` has a real event timestamp (`order_ts`), so
   > that's the natural cursor — the connector pulls orders newer than the last one
   > it saw. The other tables have no update timestamp, so their auto-increasing
   > **primary key doubles as the cursor** (these tables are insert-only in the lab,
   > so a rising ID reliably marks "new rows"). In production you'd prefer a real
   > `updated_at` column as the cursor wherever one exists.

   *(`fact_sales_forecast` is the Data Science team's forecast hand-off — you'll put
   it to the test in Lab 5.)*
6. **Destination:**
   - Catalog: **`retail_corp`**
   - Schema: **`bronze`**
7. **Schedule:** choose **Manual / Triggered** for the lab (in production you'd
   pick a cadence, e.g. hourly). Name the ingestion pipeline
   `retail_corp_bronze_ingest`.
8. Click **Create** and then **Run**.

> 🧭 **Don't see the PostgreSQL connector?** Lakeflow Connect connectors are
> enabled per workspace. If it's missing, that's an admin toggle — ask your
> workspace admin to enable the Lakeflow Connect PostgreSQL/Lakebase connector,
> then come back to this step.

When the pipeline finishes, it will have written the six tables into
`retail_corp.bronze` (which started empty after Lab 0). The next cells verify that.

In [ ]:
# 1.2-helper — connection details for the Lakeflow Connect PostgreSQL connection.
# Copy these into the "Create connection" form (1.2a). The token is the password
# and is short-lived (~1 hour) — if ingestion later fails auth, re-run this cell
# for a fresh token and update the connection.
import uuid
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
inst = w.database.get_database_instance(name=cfg["LAKEBASE_INSTANCE"])
cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()),
    instance_names=[cfg["LAKEBASE_INSTANCE"]],
)
token = getattr(cred, "token", None) or getattr(cred, "credential", None)

print("Use these in Catalog → External Data → Connections → Create connection:")
print("  Connection name : retail_corp_lakebase_conn")
print("  Connection type : PostgreSQL")
print("  Host            :", inst.read_write_dns)
print("  Port            : 5432")
print("  Database        :", cfg["LAKEBASE_DB"])
print("  User            :", w.current_user.me().user_name)
print("  Password (token):", token)
print("\n⏱️  Token expires in ~1 hour — create the connection and run the pipeline now.")

## 1.4 — Verify the bronze layer

Once the Lakeflow Connect pipeline (1.2) has run, your previously-empty bronze
schema should now hold six populated tables. As a business user, you can confirm
this **entirely in the UI** — no code needed.

### Verify in the Catalog UI
1. In the left sidebar, click **Catalog**.
2. Expand **`retail_corp`** → **`bronze`**. You should now see **six tables**:
   `dim_product`, `dim_customer`, `fact_orders`, `fact_order_items`,
   `fact_marketing_campaigns`, and `fact_sales_forecast`.
   *(Before Lab 1 this schema was empty — seeing the tables appear is proof the
   ingestion worked.)*
3. Click **`fact_orders`**, then open the **Sample Data** tab to eyeball real rows.
4. Check the **Overview** tab for the **row count** and the columns. `fact_orders`
   should have roughly **17.5K rows**; `fact_order_items` roughly **29K**.
5. Open the **Lineage** tab and confirm the table traces back to your **Lakeflow
   Connect** ingestion pipeline — this is how you'd prove *where* the data came from.

### Check the ingestion pipeline's run
6. Go to **Jobs & Pipelines** (left sidebar) and open **`retail_corp_bronze_ingest`**.
7. Confirm the latest run **succeeded** and note how many rows it wrote per table.
   This is the dashboard you'd watch in real life to know your data is fresh.

> ✅ **What "good" looks like:** all six tables present in `bronze`, each with rows,
> and a green/successful ingestion run. If a table is missing, re-run the pipeline
> from step 1.2.

## 1.5 — Bring in the market-research PDF

Your last source is the **external market-research PDF**. There are two ways it
reaches Databricks; you'll use whichever fits:

**A. It's already in your Volume (from Lab 0's deploy).** Verify with the cell below.

**B. You'll upload it into a Genie Space in Lab 5.** Lab 5 needs the PDF attached
to a Genie Space so Genie can read it. You downloaded the PDF to your laptop at the
end of Lab 0 — keep it ready. (Genie Spaces read attachments directly; you don't
need to parse the PDF here.)

For now, just confirm the file is available.

In [ ]:
pdf_dir = f"{cfg['VOLUME_PATH']}/market_research"
try:
    files = dbutils.fs.ls(pdf_dir)
    print("Files in", pdf_dir, ":")
    for f in files:
        print("  •", f.name, f"({f.size:,} bytes)")
    print("\n✓ The market-research PDF is in your Volume. In Lab 5 you'll upload")
    print("  it (from your laptop copy) into a Genie Space.")
except Exception as e:
    print("PDF not found in the Volume:", str(e)[:150])
    print("→ That's OK. You downloaded it from assets/market_research/ in Lab 0.")
    print("  You'll upload it into a Genie Space in Lab 5.")

## ✅ Lab 1 complete

You've connected your data sources:
- ✅ Understood **Lakeflow Connect** and when to use it
- ✅ Ingested the Lakebase Postgres tables into **bronze** with Lakeflow Connect
- ✅ Verified all six bronze tables went from empty to populated
- ✅ Located the market-research **PDF** for Lab 5
- ✅ Took a first rough peek at revenue by category

**Next up → Lab 2: Data Discovery.** You'll use Unity Catalog, Genie One, and
Domains to explore what data you actually have — and fix any missing metadata.